In [1]:
# cache_index.py
# FAISS-based vector index with cosine similarity (via L2-normalized inner product).

import faiss
import numpy as np
from typing import Any, Dict, Tuple, List

def _normalize(x: np.ndarray) -> np.ndarray:
    n = np.linalg.norm(x, axis=1, keepdims=True) + 1e-12
    return x / n

class CacheIndex:
    def __init__(self, dim: int):
        # Inner-product index; with L2-normalized vectors this is cosine similarity.
        self.index = faiss.IndexFlatIP(dim)
        self._payloads: List[Dict[str, Any]] = []
        self._vecs: List[np.ndarray] = []

    def add(self, vec: List[float], payload: Dict[str, Any]) -> None:
        v = np.array([vec], dtype="float32")
        v = _normalize(v)
        self.index.add(v)              # add to FAISS
        self._vecs.append(v)           # keep a copy (optional, helpful for persistence)
        self._payloads.append(payload)

    def search(self, vec: List[float], topk: int = 1) -> Tuple[float, Dict[str, Any] | None]:
        if len(self._payloads) == 0:
            return 0.0, None
        q = np.array([vec], dtype="float32")
        q = _normalize(q)
        scores, ids = self.index.search(q, topk)
        best_id = int(ids[0][0])
        best_score = float(scores[0][0])
        if best_id == -1:
            return 0.0, None
        return best_score, self._payloads[best_id]
